# Maia Lite 0.3.0 — Auditoria final de capacidades

**Objetivo:** avaliação *sem treinamento* do modelo-base e do checkpoint Full-Linear SFT, antes da exportação para Ollama. O treinamento terminou em **5744 updates** e a validação final registrada foi **2,22394**. Uma loss melhor **não implica** instruções corretas.

**Princípios:** mesmo conjunto de questões e configurações para todos os candidatos; resultados brutos preservados; métricas determinísticas separadas de julgamento humano; nenhum resultado inventado; configurações e hashes registrados. Testes sintéticos não substituem benchmarks independentes. **Não use este conjunto para ajustar o modelo e depois reportá-lo como teste cego.**

### Caminhos de execução
1. **Ollama**: testar um modelo já importado, local ou via servidor; é o caminho mais simples para uma comparação de modelos.
2. **Transformers + PEFT**: carregar o checkpoint HF e adaptador LoRA *somente se os arquivos forem compatíveis*. O checkpoint do modelo-base deve ser **exatamente o utilizado no SFT** (verifique step 300000 versus 309616). Modelos PyTorch de arquitetura personalizada precisam de um *loader* do projeto; este notebook não inventa a arquitetura.

Recomendado: Colab CPU para Ollama remoto; GPU apenas se necessário para inferência HF. Sem downloads automáticos de pesos.

In [ ]:
# Instale somente as dependências necessárias para o backend escolhido.
%pip -q install pandas numpy matplotlib requests tqdm
# Para HF/PEFT (opcional):
# %pip -q install transformers peft accelerate safetensors sentencepiece


## 1. Configuração e auditoria de ambiente

In [ ]:
from pathlib import Path
import os, sys, json, re, math, time, random, hashlib, unicodedata, statistics, platform
from datetime import datetime, timezone
import numpy as np, pandas as pd, requests
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

SEED = 20261121
random.seed(SEED); np.random.seed(SEED)
OUT = Path('/content/maia_030_final_audit') if Path('/content').exists() else Path('./maia_030_final_audit')
OUT.mkdir(parents=True, exist_ok=True)
BACKEND = 'ollama'  # 'ollama' ou 'hf_peft'
OLLAMA_URL = os.environ.get('OLLAMA_URL','http://127.0.0.1:11434')
OLLAMA_MODEL = os.environ.get('OLLAMA_MODEL','maia-lite:0.3.0')
# Configuração HF opcional: caminhos LOCAIS; não confundir checkpoint base do step 300000 com 309616.
HF_BASE = ''       # diretório HuggingFace com config.json e pesos
HF_ADAPTER = ''    # diretório PEFT com adapter_config.json
HF_TRUST_REMOTE_CODE = False  # ativar somente após auditoria do código
PROMPT_STYLE = 'qa' # 'qa', 'instruction', 'plain'; comparar estilos em teste separado
REPEATS = 3
TEMPERATURES = [0.0, 0.7]
MAX_NEW_TOKENS = 96
REQUEST_TIMEOUT_S = 180
print('Saída:', OUT.resolve(), 'Backend:', BACKEND)


## 2. Banco de provas

Questões curtas, com respostas verificáveis, sem dependência de rede. Categorias: factual, instrução estrita, lógica, aritmética, manipulação de texto, tradução, contexto, honestidade, código e resistência à repetição. Para tarefas abertas, as métricas automáticas são **diagnósticas**, não notas de verdade. O banco inclui as três falhas relatadas.

In [ ]:
TESTS = []
def add(cat, prompt, expected=None, mode='exact', lang='pt', tags=None):
    TESTS.append(dict(id=f'T{len(TESTS)+1:03}', category=cat, prompt=prompt,
                      expected=expected, mode=mode, language=lang, tags=tags or []))
# Três falhas observadas
add('factual','Qual é a capital do Brasil?','Brasília','contains')
add('instruction','Diga apenas a palavra azul.','azul','exact')
add('logic','Se Ana é mais velha que Bia, quem é mais nova?','Bia','exact')
# Factual estável
for q,a in [('Qual é a capital da França?','Paris'),('Qual é a capital da Argentina?','Buenos Aires'),('Qual é o maior planeta do Sistema Solar?','Júpiter'),('Qual é a fórmula química da água?','H2O'),('Quantos dias tem uma semana?','7'),('Qual é o idioma oficial do Brasil?','português'),('Qual é a capital de Portugal?','Lisboa'),('Qual é a capital do Japão?','Tóquio'),('Em qual continente fica o Egito?','África'),('Qual é o símbolo químico do ouro?','Au')]: add('factual',q,a,'contains')
# Instruções literais
for word in ['verde','sim','não','gato','42','OK','fim','casa','sol','teste']:
    add('instruction',f'Responda somente com: {word}',word,'exact')
add('instruction','Não explique. Escreva exatamente três letras: ABC.','ABC','exact')
add('instruction','Escreva somente o número 5.','5','exact')
add('instruction','Responda somente com a palavra: amarelo.','amarelo','exact')
# Relações / lógica
for q,a in [('João é mais alto que Pedro. Quem é mais baixo?','Pedro'),('Maria chegou antes de Clara. Quem chegou depois?','Clara'),('Se Paulo é irmão de Luiza, Luiza é irmã de quem?','Paulo'),('O livro está dentro da caixa. Onde está o livro?','caixa'),('Se todos os gatos são animais e Mimi é um gato, Mimi é um animal?','sim'),('Se A > B e B > C, quem é maior: A ou C?','A'),('O carro vermelho é mais rápido que o azul. Qual é mais lento?','azul'),('Lia tem menos moedas que Rui. Quem tem mais moedas?','Rui'),('A caixa A pesa 2 kg e a B pesa 5 kg. Qual é mais pesada?','B'),('Se hoje é terça-feira, que dia é amanhã?','quarta-feira')]: add('logic',q,a,'contains')
# Matemática com resposta única
for a,b in [(2,3),(7,8),(12,15),(31,9),(40,17),(13,21),(6,6),(19,23)]: add('math',f'Quanto é {a} + {b}? Responda só com o número.',str(a+b),'exact')
for a,b in [(9,4),(20,7),(50,18),(17,9),(100,37)]: add('math',f'Quanto é {a} - {b}? Responda só com o número.',str(a-b),'exact')
for a,b in [(3,4),(7,6),(8,9),(12,5),(11,11)]: add('math',f'Quanto é {a} vezes {b}? Responda só com o número.',str(a*b),'exact')
for a,b in [(20,5),(81,9),(36,6),(144,12)]: add('math',f'Quanto é {a} dividido por {b}? Responda só com o número.',str(a//b),'exact')
# Transformações de texto
for word in ['casa','maia','python','brasil','modelo','janela','computador','escola']:
    add('text',f'Escreva em MAIÚSCULAS somente a palavra {word}.',word.upper(),'exact')
for word in ['CASA','GATO','MAIA','LIVRO','VERDE']:
    add('text',f'Escreva em minúsculas somente {word}.',word.lower(),'exact')
# Tradução controlada
for pt,en in [('casa','house'),('gato','cat'),('água','water'),('livro','book'),('sol','sun'),('escola','school')]:
    add('translation',f'Traduza para inglês, escrevendo só a palavra: {pt}',en,'exact')
    add('translation',f'Translate to Portuguese. Output one word: {en}',pt,'exact','en')
for es,pt in [('casa','casa'),('perro','cachorro'),('libro','livro'),('agua','água')]:
    add('translation',f'Traduce al portugués. Escribe solo una palabra: {es}',pt,'exact','es')
# Contexto fornecido: mede uso da informação do prompt, não conhecimento enciclopédico
for name,city in [('Nina','Recife'),('Otávio','Manaus'),('Lara','Belém'),('Caio','Curitiba'),('Sofia','Natal'),('Breno','Fortaleza')]:
    add('context',f'Informação: {name} mora em {city}. Pergunta: Em qual cidade {name} mora? Responda somente a cidade.',city,'exact')
# Honestidade e seguimento de limites
for q in ['Qual é a senha que estou pensando agora?','Qual é o número secreto que não lhe contei?','Qual é o nome do meu vizinho, sem eu informar?']:
    add('uncertainty',q,None,'human')
# Código: revisão humana, não executar código produzido sem sandbox
for q in ['Escreva uma função Python que some dois inteiros.','Escreva uma função Python que retorne o maior de dois números.','Escreva um programa Python que imprima Olá, mundo!','Escreva uma função Python que teste se um número é par.']:
    add('code',q,None,'human')
# Fluência e repetição: revisão humana e métricas de repetição
for q in ['Explique em três frases o que é uma bateria recarregável.','Explique a diferença entre CPU e GPU em duas frases.','Escreva um parágrafo coerente sobre o ciclo da água.','Explique brevemente o que é uma rede neural.']:
    add('fluency',q,None,'human')
# Inglês e espanhol com alvos inequívocos
for q,a,lang in [('What is the capital of Brazil?','Brasília','en'),('Say only the word blue.','blue','en'),('If Alice is older than Bob, who is younger?','Bob','en'),('¿Cuál es la capital de Brasil?','Brasília','es'),('Di solamente la palabra azul.','azul','es'),('Si Ana es mayor que Bia, ¿quién es menor?','Bia','es')]: add('multilingual',q,a,'exact' if 'only' in q or 'solamente' in q or 'younger' in q or 'menor' in q else 'contains',lang)

assert len({x['id'] for x in TESTS}) == len(TESTS)
print('Total de testes:',len(TESTS))
print(pd.Series([x['category'] for x in TESTS]).value_counts().to_string())
(OUT/'test_bank.json').write_text(json.dumps(TESTS,ensure_ascii=False,indent=2),encoding='utf-8')


## 3. Backends de inferência: Ollama ou Hugging Face/PEFT

In [ ]:
class OllamaBackend:
    def __init__(self, url, model):
        self.url=url.rstrip('/'); self.model=model
        r=requests.get(self.url+'/api/tags',timeout=15); r.raise_for_status()
        available=[m.get('name','') for m in r.json().get('models',[])]
        if model not in available:
            raise RuntimeError(f'Modelo {model!r} não encontrado em Ollama. Disponíveis: {available}')
    def generate(self,prompt,temperature,seed,max_tokens):
        payload={'model':self.model,'prompt':prompt,'stream':False,'options':{
            'temperature':temperature,'seed':seed,'num_predict':max_tokens,
            'num_ctx':2048,'repeat_penalty':1.0}}
        r=requests.post(self.url+'/api/generate',json=payload,timeout=REQUEST_TIMEOUT_S)
        r.raise_for_status(); d=r.json()
        return d.get('response',''), {'eval_count':d.get('eval_count'), 'eval_duration':d.get('eval_duration')}

class HFBackend:
    def __init__(self,base,adapter):
        if not base or not Path(base).is_dir():
            raise ValueError('Defina HF_BASE com um diretório HuggingFace válido. Checkpoint .pt customizado requer loader do projeto.')
        from transformers import AutoTokenizer, AutoModelForCausalLM
        import torch
        self.torch=torch
        self.tokenizer=AutoTokenizer.from_pretrained(base,trust_remote_code=HF_TRUST_REMOTE_CODE)
        self.model=AutoModelForCausalLM.from_pretrained(base,trust_remote_code=HF_TRUST_REMOTE_CODE,
             torch_dtype='auto',device_map='auto',low_cpu_mem_usage=True)
        if adapter:
            from peft import PeftModel
            self.model=PeftModel.from_pretrained(self.model,adapter,is_trainable=False)
        self.model.eval()
        self.model_name=base+' + '+adapter
    def generate(self,prompt,temperature,seed,max_tokens):
        torch=self.torch
        torch.manual_seed(seed)
        x=self.tokenizer(prompt,return_tensors='pt').to(self.model.device)
        if x['input_ids'].shape[-1]>=2048:
            raise ValueError('Prompt excede o contexto 2048')
        kwargs=dict(max_new_tokens=max_tokens,do_sample=temperature>0,
                    pad_token_id=self.tokenizer.eos_token_id)
        if temperature>0: kwargs['temperature']=temperature
        with torch.inference_mode(): y=self.model.generate(**x,**kwargs)
        txt=self.tokenizer.decode(y[0,x['input_ids'].shape[-1]:],skip_special_tokens=True)
        return txt,{'eval_count':int(y.shape[-1]-x['input_ids'].shape[-1])}

def make_backend():
    if BACKEND=='ollama': return OllamaBackend(OLLAMA_URL,OLLAMA_MODEL)
    if BACKEND=='hf_peft': return HFBackend(HF_BASE,HF_ADAPTER)
    raise ValueError('BACKEND deve ser ollama ou hf_peft')

# Ative apenas depois de conferir os caminhos e o modelo:
backend=make_backend()
print('Backend pronto:',BACKEND)


### Atenção: prompt e formatação

O prompt deve reproduzir o formato utilizado no SFT. Um modelo treinado com `### Instrução/### Resposta`, por exemplo, pode falhar com `P:/R:` por incompatibilidade de template. **Teste primeiro os estilos; não escolha o melhor estilo depois de olhar a nota final sem registrar essa seleção.** O backend Ollama usa `/api/generate` com `raw=False` por padrão, portanto pode aplicar o template do Modelfile: verifique com `ollama show --modelfile` e evite dupla formatação.

In [ ]:
def format_prompt(q,style):
    if style=='qa': return f'P: {q}\nR:'
    if style=='instruction': return f'### Instrução:\n{q}\n\n### Resposta:\n'
    if style=='plain': return q
    raise ValueError(style)

def normalize(s):
    s=unicodedata.normalize('NFKD',s.lower().strip())
    s=''.join(c for c in s if not unicodedata.combining(c))
    s=re.sub(r'[^a-z0-9]+',' ',s).strip()
    return s

def evaluate(test,response):
    mode=test['mode']; exp=test['expected']
    if mode=='human': return None
    a=normalize(response); b=normalize(exp)
    if mode=='exact': return int(a==b)
    if mode=='contains': return int(bool(re.search(r'(?<!\w)'+re.escape(b)+r'(?!\w)',a)))
    raise ValueError(mode)

def repetition_metrics(text):
    toks=normalize(text).split()
    if not toks:return {'repeat_3gram_rate':0.,'distinct_2':0.,'n_words':0}
    tri=[tuple(toks[i:i+3]) for i in range(max(0,len(toks)-2))]
    bi=[tuple(toks[i:i+2]) for i in range(max(0,len(toks)-1))]
    return {'repeat_3gram_rate':1-len(set(tri))/len(tri) if tri else 0.,
            'distinct_2':len(set(bi))/len(bi) if bi else 0.,'n_words':len(toks)}


## 4. Teste de fumaça (não executar bateria inteira antes de validar o template)

In [ ]:
for test in TESTS[:3]:
    prompt=format_prompt(test['prompt'],PROMPT_STYLE)
    answer,meta=backend.generate(prompt,0.0,SEED,64)
    print('P:',test['prompt'],'\nR:',repr(answer),'\nEsperado:',test['expected'],'Acerto:',evaluate(test,answer),'\n')


## 5. Bateria completa, repetições e checkpoint incremental

In [ ]:
# Execução: ~len(TESTS) * len(TEMPERATURES) * REPEATS chamadas.
# Para economia, começar com TEMPERATURES=[0.0] e REPEATS=1; depois executar o protocolo completo.
RUN_NAME = f"{BACKEND}_{OLLAMA_MODEL if BACKEND=='ollama' else 'hf'}_{PROMPT_STYLE}".replace('/','_').replace(':','_')
RESULT_PATH=OUT/f'{RUN_NAME}_results.jsonl'
print('Chamadas previstas:',len(TESTS)*len(TEMPERATURES)*REPEATS)
print('Arquivo:',RESULT_PATH)

def run_all():
    seen=set()
    if RESULT_PATH.exists():
        for line in RESULT_PATH.read_text(encoding='utf-8').splitlines():
            if line.strip():
                try:
                    d=json.loads(line); seen.add((d['id'],d['temperature'],d['repeat']))
                except (ValueError,KeyError): pass
    tasks=[(t,temp,rep) for t in TESTS for temp in TEMPERATURES for rep in range(REPEATS)
           if (t['id'],temp,rep) not in seen]
    with RESULT_PATH.open('a',encoding='utf-8') as f:
        for t,temp,rep in tqdm(tasks,desc='Auditoria Maia'):
            prompt=format_prompt(t['prompt'],PROMPT_STYLE)
            started=time.monotonic()
            try:
                answer,extra=backend.generate(prompt,temp,SEED+rep,MAX_NEW_TOKENS)
                err=None
            except Exception as e:
                answer='';extra={};err=repr(e)
            row={**t,'prompt_formatted':prompt,'answer':answer,'score':None if err else evaluate(t,answer),
                 'temperature':temp,'repeat':rep,'seed':SEED+rep,'elapsed_s':round(time.monotonic()-started,3),
                 'error':err,'meta':extra,**repetition_metrics(answer)}
            f.write(json.dumps(row,ensure_ascii=False)+'\n');f.flush()
    print('Concluído:',RESULT_PATH)

# Descomente para executar a avaliação completa:
# run_all()


## 6. Análise quantitativa: acertos, estabilidade e falhas

In [ ]:
if not RESULT_PATH.exists():
    print('Ainda não há resultados. Execute run_all() na célula anterior.')
else:
    df=pd.read_json(RESULT_PATH,lines=True)
    valid=df[df.error.isna()].copy()
    scored=valid[valid.score.notna()].copy()
    print('Execuções:',len(df),'Erros:',df.error.notna().sum(),'Testes objetivos:',len(scored))
    if len(scored):
        print('\nTaxa de acerto por categoria e temperatura:')
        display(scored.groupby(['temperature','category']).score.agg(['mean','count']).round(3))
        print('\nTaxa global (macro por categoria, não ponderada):')
        print(scored.groupby(['temperature','category']).score.mean().groupby(level=0).mean().round(3))
        pivot=scored.groupby(['category','temperature']).score.mean().unstack()
        pivot.plot(kind='bar',ylim=(0,1),figsize=(11,4),title='Acerto objetivo por categoria')
        plt.ylabel('Fração de acertos');plt.tight_layout();plt.show()
        print('\nFalhas objetivas (amostra):')
        display(scored[scored.score==0][['id','category','prompt','expected','answer','temperature']].head(30))
    print('\nIndicadores de repetição (descritivos):')
    display(valid.groupby(['temperature','category'])[['repeat_3gram_rate','distinct_2','n_words']].mean().round(3))
    stability=(valid.groupby(['id','temperature']).answer.nunique().rename('n_answers').reset_index())
    print('\nVariação de respostas por questão/temperatura:')
    display(stability.groupby('temperature').n_answers.agg(['mean','max']).round(2))
    scored.to_csv(OUT/f'{RUN_NAME}_scored.csv',index=False)


## 7. Revisão humana cega (não forçar avaliação automática)

As questões de código, fluência e incerteza exigem avaliação humana. Uma resposta como “não tenho acesso à senha” é apropriada, mas expressões equivalentes não devem ser penalizadas por um teste exato. Avalie **correção (0–2), aderência à instrução (0–2), clareza (0–2), ausência de repetição (0–2)**. Em código, avalie também sintaxe e comportamento, executando somente em ambiente isolado.

In [ ]:
if RESULT_PATH.exists():
    df=pd.read_json(RESULT_PATH,lines=True)
    human=df[(df['mode']=='human') & df.error.isna()].copy()
    human=human[['id','category','language','prompt','answer','temperature','repeat']].copy()
    for col in ['correcao_0_2','instrucao_0_2','clareza_0_2','sem_repeticao_0_2','observacoes']:
        human[col]=''
    human.to_csv(OUT/f'{RUN_NAME}_human_review.csv',index=False)
    print('Planilha para revisão humana:',OUT/f'{RUN_NAME}_human_review.csv')
else: print('Execute a bateria primeiro.')


## 8. Comparação entre checkpoints (sem mudar o protocolo)

Repita a bateria com os mesmos `TESTS`, `PROMPT_STYLE`, `TEMPERATURES`, `REPEATS` e `SEED`, mudando **apenas** `OLLAMA_MODEL` ou `HF_ADAPTER` e instanciando novamente `backend`. Guarde os resultados em arquivos separados. Para comparar base e SFT, use a mesma revisão do checkpoint-base que originou o adaptador. Para comparar com modelos TinyStories/BabyLM, separe tarefas fora do domínio e não conclua que fluência narrativa equivale a capacidade geral.

**Interpretação:** pontuações automáticas são aproximações. Não conclua que uma resposta factual incorreta significa que o conhecimento está ausente; compare também variações de template e decodificação. O inverso também vale: acertos por memorização não demonstram raciocínio.

In [ ]:
files=sorted(OUT.glob('*_results.jsonl'))
rows=[]
for path in files:
    d=pd.read_json(path,lines=True)
    d=d[d.error.isna() & d.score.notna()]
    for (cat,temp),group in d.groupby(['category','temperature']):
        rows.append({'modelo_execucao':path.stem,'categoria':cat,'temperatura':temp,
                     'acerto':group.score.mean(),'n':len(group)})
comparison=pd.DataFrame(rows)
if len(comparison):
    display(comparison.sort_values(['categoria','modelo_execucao']))
    comparison.to_csv(OUT/'comparison.csv',index=False)
else: print('Sem execuções comparáveis.')


## 9. Manifesto reproduzível e empacotamento

In [ ]:
import zipfile
manifest={
  'project':'Maia Lite 0.3.0 final capability audit',
  'created_utc':datetime.now(timezone.utc).isoformat(),
  'python':sys.version,'platform':platform.platform(),
  'backend':BACKEND,'model':OLLAMA_MODEL if BACKEND=='ollama' else HF_BASE,
  'adapter':HF_ADAPTER if BACKEND=='hf_peft' else None,
  'prompt_style':PROMPT_STYLE,'temperatures':TEMPERATURES,'repeats':REPEATS,
  'seed':SEED,'max_new_tokens':MAX_NEW_TOKENS,
  'test_count':len(TESTS),
  'test_bank_sha256':hashlib.sha256((OUT/'test_bank.json').read_bytes()).hexdigest(),
  'training_final_update':5744,'training_final_val_loss':2.22394,
  'limitations':'This notebook does not attest checkpoint provenance, dataset independence, or model quality until executed and reviewed.'
}
(OUT/f'{RUN_NAME}_manifest.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
zip_path=OUT.parent/'maia_030_final_audit_results.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT.glob('*'):
        if p.is_file():z.write(p,arcname=p.name)
print('Relatório:',zip_path)
# Colab: descomente para baixar
# from google.colab import files
# files.download(str(zip_path))


## 10. Critérios para encerrar a versão

Registre separadamente: (a) taxa de acerto por categoria, (b) sensibilidade ao template, (c) estabilidade sob múltiplas sementes, (d) qualidade humana e (e) limitações de inferência/empacotamento. **Não use um único percentual como classificação global do modelo.**

O objetivo não é salvar a reputação do 0.3.0, mas deixar um registro científico verificável: o que o modelo realmente faz, o que não faz e em quais condições. A exportação GGUF/Ollama deve ser validada com as mesmas perguntas para detectar regressões de conversão ou quantização.